# Urban Data Explorer - EDA, nettoyage et fusion

Ce notebook sert de base de travail pour :
- explorer les CSV disponibles,
- nettoyer les jeux de données,
- harmoniser les clés géographiques,
- préparer des tables fusionnées prêtes pour l'API et le dashboard.

Flux de travail cible :
1. Bronze: fichiers bruts.
2. Silver: données nettoyées et normalisées.
3. Gold: tables agrégées et fusionnées pour la dataviz.

In [1]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

DATA_DIR = Path("csv")
RAW_FILES = {
    "arrondissements": DATA_DIR / "arrondissements.csv",
    "iris": DATA_DIR / "iris.csv",
    "transactions": DATA_DIR / "75.csv",
    "social_housing": DATA_DIR / "logements-sociaux-finances-a-paris.csv",
    "rent_caps": DATA_DIR / "logement-encadrement-des-loyers.csv",
    "territorial_stats": DATA_DIR / "donnee-data.gouv-2025-geographie2025-produit-le2026-02-03.csv",
}

print("CSV trouvés:")
for name, path in RAW_FILES.items():
    print(f"- {name}: {path} ({'OK' if path.exists() else 'MANQUANT'})")

CSV trouvés:
- arrondissements: csv\arrondissements.csv (OK)
- iris: csv\iris.csv (OK)
- transactions: csv\75.csv (OK)
- social_housing: csv\logements-sociaux-finances-a-paris.csv (OK)
- rent_caps: csv\logement-encadrement-des-loyers.csv (OK)
- territorial_stats: csv\donnee-data.gouv-2025-geographie2025-produit-le2026-02-03.csv (OK)


In [3]:
def normalize_text(value: str) -> str:
    if pd.isna(value):
        return value
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = value.lower().strip()
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    renamed = {}
    for column in df.columns:
        clean = normalize_text(column)
        clean = clean.replace("'", "")
        clean = re.sub(r"[^a-z0-9]+", "_", clean)
        clean = re.sub(r"_+", "_", clean).strip("_")
        renamed[column] = clean
    return df.rename(columns=renamed)


def read_csv_safely(path: Path, **kwargs) -> pd.DataFrame:
    defaults = {"sep": ";", "encoding": "utf-8", "low_memory": False}
    defaults.update(kwargs)
    return pd.read_csv(path, **defaults)


def profile_dataframe(df: pd.DataFrame, name: str) -> None:
    print(f"\n===== {name} =====")
    print("shape:", df.shape)
    print("colonnes:")
    print(list(df.columns))
    display(df.head(3))
    display(df.isna().mean().sort_values(ascending=False).head(12).to_frame("taux_manquants"))

In [6]:
arrondissements_raw = read_csv_safely(RAW_FILES["arrondissements"])
iris_raw = read_csv_safely(RAW_FILES["iris"])
transactions_raw = read_csv_safely(RAW_FILES["transactions"], sep=",")
social_housing_raw = read_csv_safely(RAW_FILES["social_housing"])
rent_caps_raw = read_csv_safely(RAW_FILES["rent_caps"])

profile_dataframe(arrondissements_raw, "Arrondissements")
profile_dataframe(iris_raw, "IRIS")
profile_dataframe(transactions_raw, "Transactions immobilières")
profile_dataframe(social_housing_raw, "Logements sociaux financés")
profile_dataframe(rent_caps_raw, "Encadrement des loyers")


===== Arrondissements =====
shape: (20, 10)
colonnes:
['Identifiant séquentiel de l’arrondissement', 'Numéro d’arrondissement', 'Numéro d’arrondissement INSEE', 'Nom de l’arrondissement', 'Nom officiel de l’arrondissement', 'N_SQ_CO', 'Surface', 'Périmètre', 'Geometry X Y', 'Geometry']


,Identifiant séquentiel de l’arrondissement,Numéro d’arrondissement,Numéro d’arrondissement INSEE,Nom de l’arrondissement,Nom officiel de l’arrondissement,N_SQ_CO,Surface,Périmètre,Geometry X Y,Geometry
0,750000001,1,75101,1er Ardt,Louvre,750001537,1.824613e+06,6054.936862,"48.862562701836005, 2.3364433620533878","{""coordinates"": [[[2.328007329038849, 48.86991742140714], [2.329965588686571, 48.868514169174276], [2.33030679532087..."
1,750000011,11,75111,11ème Ardt,Popincourt,750001537,3.665442e+06,8282.011886,"48.859059221342484, 2.380058308197898","{""coordinates"": [[[2.3962365763098292, 48.85415458748718], [2.3970750354459907, 48.853082331641744], [2.397117501448..."
2,750000014,14,75114,14ème Ardt,Observatoire,750001537,5.614877e+06,10317.483310,"48.82924450048987, 2.3265420441989466","{""coordinates"": [[[2.333806501627019, 48.84060921979979], [2.336728982141496, 48.83965349513613], [2.339690927267829..."


,taux_manquants
Identifiant séquentiel de l’arrondissement,0.0
Numéro d’arrondissement,0.0
Numéro d’arrondissement INSEE,0.0
Nom de l’arrondissement,0.0
Nom officiel de l’arrondissement,0.0
N_SQ_CO,0.0
Surface,0.0
Périmètre,0.0
Geometry X Y,0.0
Geometry,0.0



===== IRIS =====
shape: (992, 10)
colonnes:
['Geo Shape', 'DEP', 'INSEE_COM', 'NOM_COM', 'IRIS', 'CODE_IRIS', 'NOM_IRIS', 'TYP_IRIS', 'Geo Point', 'ID']


,Geo Shape,DEP,INSEE_COM,NOM_COM,IRIS,CODE_IRIS,NOM_IRIS,TYP_IRIS,Geo Point,ID
0,"{""coordinates"": [[[2.348444771796439, 48.862379879643505], [2.347747960057199, 48.860986592099664], [2.3476618237374...",75,75101,Paris 1er Arrondissement,205,751010205,Les Halles 5,A,"48.862297570166575, 2.34534858519305",IRIS____0000000751010205
1,"{""coordinates"": [[[2.339143935838259, 48.86220567061473], [2.339469249651607, 48.86213461940564], [2.339130016712214...",75,75101,Paris 1er Arrondissement,303,751010303,Palais Royal 3,A,"48.862970884171986, 2.335627404567633",IRIS____0000000751010303
2,"{""coordinates"": [[[2.295542880656993, 48.87330312959988], [2.29620959727898, 48.87249404502659], [2.297501223263732,...",75,75116,Paris 16e Arrondissement,6411,751166411,Chaillot 11,A,"48.871380349816135, 2.296118003770666",IRIS____0000000751166411


...

,Identifiant livraison,Adresse du programme,Code postal,Ville,Année du financement - agrément,Bailleur social,Nombre total de logements financés,Dont nombre de logements PLA I,Dont nombre de logements PLUS,Dont nombre de logements PLUS CD,Dont nombre de logements PLS,Mode de réalisation,Commentaires,Arrondissement,Nature de programme,Coordonnée en X (L93),Coordonnée en Y (L93),geo_shape,geo_point_2d
0,2000011,16-20 RUE DES MEUNIERS,75012,Paris,2001,RES.URB.,62,0,62,0,0,acquisition conventionnement,NaN,12,logement familial,655823.1529,6.859495e+06,"{""coordinates"": [2.398175442412243, 48.83399817959342], ""type"": ""Point""}","48.83399817959342, 2.398175442412243"
1,2000035,25 RUE DES ANNELETS,75019,Paris,2001,RIVP,18,10,8,0,0,acquisition réhabilitation,NaN,19,logement familial,655225.8330,6.864433e+06,"{""coordinates"": [2.3895183752327043, 48.8783619998301], ""type"": ""Point""}","48.8783619998301, 2.3895183752327043"
2,2000044,LOT D1B - 86 RUE DES HAIES,75020,Paris,2001,RIVP,6,0,6,0,0,acquisition réhabilitation,NaN,20,logement familial,656258.4385,6.861828e+06,"{""coordinates"": [2.403865260493333, 48.85500366971155], ""type"": ""Point""}","48.85500366971155, 2.403865260493333"


,taux_manquants
Commentaires,1.0
Adresse du programme,0.0
Identifiant livraison,0.0
Code postal,0.0
Ville,0.0
Bailleur social,0.0
Année du financement - agrément,0.0
Dont nombre de logements PLA I,0.0
Dont nombre de logements PLUS,0.0
Dont nombre de logements PLUS CD,0.0



===== Encadrement des loyers =====
shape: (17920, 14)
colonnes:
['Année', 'Secteurs géographiques', 'Numéro du quartier', 'Nom du quartier', 'Nombre de pièces principales', 'Epoque de construction', 'Type de location', 'Loyers de référence', 'Loyers de référence majorés', 'Loyers de référence minorés', 'Ville', 'Numéro INSEE du quartier', 'geo_shape', 'geo_point_2d']


,Année,Secteurs géographiques,Numéro du quartier,Nom du quartier,Nombre de pièces principales,Epoque de construction,Type de location,Loyers de référence,Loyers de référence majorés,Loyers de référence minorés,Ville,Numéro INSEE du quartier,geo_shape,geo_point_2d
0,2023,11,77,Belleville,3,1946-1970,non meublé,19.8,23.8,13.9,PARIS,7512077,"{""coordinates"": [[[2.3832266916750324, 48.86709755024514], [2.383136036636736, 48.86707551238079], [2.38309592865078...","48.87153120058614, 2.3875492398498035"
1,2023,2,21,Monnaie,2,Avant 1946,meublé,33.0,39.6,23.1,PARIS,7510621,"{""coordinates"": [[[2.3431685504395383, 48.851388550003655], [2.3426778705512987, 48.850357652304076], [2.34130585569...","48.85438440363996, 2.340035371130514"
2,2023,4,12,Sainte-Avoie,1,1971-1990,meublé,34.8,41.8,24.4,PARIS,7510312,"{""coordinates"": [[[2.3582176804342043, 48.86122492296987], [2.356903330605401, 48.86006582105499], [2.35686231490314...","48.862557245043206, 2.354851518245669"


,taux_manquants
Année,0.0
Secteurs géographiques,0.0
Numéro du quartier,0.0
Nom du quartier,0.0
Nombre de pièces principales,0.0
Epoque de construction,0.0
Type de location,0.0
Loyers de référence,0.0
Loyers de référence majorés,0.0
Loyers de référence minorés,0.0


## 1. Standardisation et nettoyage

Objectif: produire des tables Silver homogènes.

Règles appliquées:
- uniformiser les noms de colonnes,
- convertir les nombres et les dates,
- extraire des clés géographiques simples,
- garder une géométrie ou une coordonnée exploitable,
- supprimer les doublons et les valeurs aberrantes évidentes.

In [7]:
arrondissements = normalize_columns(arrondissements_raw.copy())
iris = normalize_columns(iris_raw.copy())
transactions = normalize_columns(transactions_raw.copy())
social_housing = normalize_columns(social_housing_raw.copy())
rent_caps = normalize_columns(rent_caps_raw.copy())

arrondissements["arrondissement_num"] = arrondissements["numero_d_arrondissement"].astype(str).str.zfill(2)
arrondissements["arrondissement_code_insee"] = arrondissements["numero_d_arrondissement_insee"].astype(str)
arrondissements["arrondissement_label"] = arrondissements["nom_officiel_de_l_arrondissement"].astype(str)

iris["code_iris"] = iris["code_iris"].astype(str)
iris["arrondissement_code_insee"] = iris["insee_com"].astype(str).str[:5]
iris["arrondissement_num"] = iris["arrondissement_code_insee"].str[-2:]
iris["iris_label"] = iris["nom_iris"].astype(str)
iris["iris_type"] = iris["typ_iris"].astype(str)

arrondissements_clean = arrondissements[[
    "arrondissement_num",
    "arrondissement_code_insee",
    "arrondissement_label",
    "surface",
    "perimetre",
    "geometry_x_y",
    "geometry",
]].drop_duplicates(subset=["arrondissement_code_insee"])

iris_clean = iris[[
    "arrondissement_num",
    "arrondissement_code_insee",
    "code_iris",
    "iris_label",
    "iris_type",
    "geo_point",
    "geo_shape",
]].drop_duplicates(subset=["code_iris"])

profile_dataframe(arrondissements_clean, "Arrondissements nettoyés")
profile_dataframe(iris_clean, "IRIS nettoyés")


===== Arrondissements nettoyés =====
shape: (20, 7)
colonnes:
['arrondissement_num', 'arrondissement_code_insee', 'arrondissement_label', 'surface', 'perimetre', 'geometry_x_y', 'geometry']


,arrondissement_num,arrondissement_code_insee,arrondissement_label,surface,perimetre,geometry_x_y,geometry
0,01,75101,Louvre,1.824613e+06,6054.936862,"48.862562701836005, 2.3364433620533878","{""coordinates"": [[[2.328007329038849, 48.86991742140714], [2.329965588686571, 48.868514169174276], [2.33030679532087..."
1,11,75111,Popincourt,3.665442e+06,8282.011886,"48.859059221342484, 2.380058308197898","{""coordinates"": [[[2.3962365763098292, 48.85415458748718], [2.3970750354459907, 48.853082331641744], [2.397117501448..."
2,14,75114,Observatoire,5.614877e+06,10317.483310,"48.82924450048987, 2.3265420441989466","{""coordinates"": [[[2.333806501627019, 48.84060921979979], [2.336728982141496, 48.83965349513613], [2.339690927267829..."


,taux_manquants
arrondissement_num,0.0
arrondissement_code_insee,0.0
arrondissement_label,0.0
surface,0.0
perimetre,0.0
geometry_x_y,0.0
geometry,0.0



===== IRIS nettoyés =====
shape: (992, 7)
colonnes:
['arrondissement_num', 'arrondissement_code_insee', 'code_iris', 'iris_label', 'iris_type', 'geo_point', 'geo_shape']


,arrondissement_num,arrondissement_code_insee,code_iris,iris_label,iris_type,geo_point,geo_shape
0,01,75101,751010205,Les Halles 5,A,"48.862297570166575, 2.34534858519305","{""coordinates"": [[[2.348444771796439, 48.862379879643505], [2.347747960057199, 48.860986592099664], [2.3476618237374..."
1,01,75101,751010303,Palais Royal 3,A,"48.862970884171986, 2.335627404567633","{""coordinates"": [[[2.339143935838259, 48.86220567061473], [2.339469249651607, 48.86213461940564], [2.339130016712214..."
2,16,75116,751166411,Chaillot 11,A,"48.871380349816135, 2.296118003770666","{""coordinates"": [[[2.295542880656993, 48.87330312959988], [2.29620959727898, 48.87249404502659], [2.297501223263732,..."


,taux_manquants
arrondissement_num,0.0
arrondissement_code_insee,0.0
code_iris,0.0
iris_label,0.0
iris_type,0.0
geo_point,0.0
geo_shape,0.0


In [8]:
transactions["date_mutation"] = pd.to_datetime(transactions["date_mutation"], errors="coerce")
transactions["valeur_fonciere"] = pd.to_numeric(transactions["valeur_fonciere"], errors="coerce")
transactions["surface_reelle_bati"] = pd.to_numeric(transactions["surface_reelle_bati"], errors="coerce")
transactions["nombre_pieces_principales"] = pd.to_numeric(transactions["nombre_pieces_principales"], errors="coerce")
transactions["longitude"] = pd.to_numeric(transactions["longitude"], errors="coerce")
transactions["latitude"] = pd.to_numeric(transactions["latitude"], errors="coerce")
transactions["type_local"] = transactions["type_local"].astype(str)
transactions["arrondissement_num"] = transactions["code_commune"].astype(str).str[-2:]
transactions["annee"] = transactions["date_mutation"].dt.year
transactions["prix_m2"] = np.where(
    transactions["surface_reelle_bati"] > 0,
    transactions["valeur_fonciere"] / transactions["surface_reelle_bati"],
    np.nan,
)

transactions_clean = transactions.loc[
    transactions["valeur_fonciere"].notna()
    & transactions["arrondissement_num"].isin([f"{i:02d}" for i in range(1, 21)])
].copy()

transactions_clean = transactions_clean[
    [
        "id_mutation",
        "date_mutation",
        "annee",
        "arrondissement_num",
        "code_commune",
        "nom_commune",
        "type_local",
        "surface_reelle_bati",
        "nombre_pieces_principales",
        "valeur_fonciere",
        "prix_m2",
        "longitude",
        "latitude",
    ]
].drop_duplicates()

transactions_clean = transactions_clean.loc[
    transactions_clean["prix_m2"].between(500, 50000) | transactions_clean["prix_m2"].isna()
]

profile_dataframe(transactions_clean, "Transactions nettoyées")


===== Transactions nettoyées =====
shape: (62446, 13)
colonnes:
['id_mutation', 'date_mutation', 'annee', 'arrondissement_num', 'code_commune', 'nom_commune', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales', 'valeur_fonciere', 'prix_m2', 'longitude', 'latitude']


,id_mutation,date_mutation,annee,arrondissement_num,code_commune,nom_commune,type_local,surface_reelle_bati,nombre_pieces_principales,valeur_fonciere,prix_m2,longitude,latitude
0,2021-1682469,2021-01-05,2021,08,75108,Paris 8e Arrondissement,Appartement,111.0,5.0,1480000.0,13333.333333,2.320982,48.876954
1,2021-1682469,2021-01-05,2021,08,75108,Paris 8e Arrondissement,Dépendance,NaN,0.0,1480000.0,NaN,2.320982,48.876954
4,2021-1682470,2021-01-05,2021,08,75108,Paris 8e Arrondissement,Dépendance,NaN,0.0,20000.0,NaN,2.319574,48.878074


,taux_manquants
surface_reelle_bati,0.379047
prix_m2,0.379047
nombre_pieces_principales,0.006982
type_local,0.006838
latitude,0.002130
longitude,0.002130
code_commune,0.000000
annee,0.000000
date_mutation,0.000000
id_mutation,0.000000


In [9]:
social_housing["annee_financement_agrement"] = pd.to_numeric(
    social_housing["annee_du_financement_agrement"], errors="coerce"
)
social_housing["nombre_total_de_logements_finances"] = pd.to_numeric(
    social_housing["nombre_total_de_logements_finances"], errors="coerce"
)
social_housing["dont_nombre_de_logements_pla_i"] = pd.to_numeric(
    social_housing["dont_nombre_de_logements_pla_i"], errors="coerce"
)
social_housing["dont_nombre_de_logements_plus"] = pd.to_numeric(
    social_housing["dont_nombre_de_logements_plus"], errors="coerce"
)
social_housing["dont_nombre_de_logements_plus_cd"] = pd.to_numeric(
    social_housing["dont_nombre_de_logements_plus_cd"], errors="coerce"
)
social_housing["dont_nombre_de_logements_pls"] = pd.to_numeric(
    social_housing["dont_nombre_de_logements_pls"], errors="coerce"
)
social_housing["arrondissement_num"] = social_housing["arrondissement"].astype(str).str.zfill(2)
social_housing["programme_type"] = social_housing["nature_de_programme"].astype(str)
social_housing["bailleur_social"] = social_housing["bailleur_social"].astype(str)

social_housing_clean = social_housing[[
    "identifiant_livraison",
    "annee_financement_agrement",
    "arrondissement_num",
    "bailleur_social",
    "programme_type",
    "nombre_total_de_logements_finances",
    "dont_nombre_de_logements_pla_i",
    "dont_nombre_de_logements_plus",
    "dont_nombre_de_logements_plus_cd",
    "dont_nombre_de_logements_pls",
    "geo_point_2d",
    "geo_shape",
]].drop_duplicates(subset=["identifiant_livraison"])

profile_dataframe(social_housing_clean, "Logements sociaux nettoyés")


===== Logements sociaux nettoyés =====
shape: (4141, 12)
colonnes:
['identifiant_livraison', 'annee_financement_agrement', 'arrondissement_num', 'bailleur_social', 'programme_type', 'nombre_total_de_logements_finances', 'dont_nombre_de_logements_pla_i', 'dont_nombre_de_logements_plus', 'dont_nombre_de_logements_plus_cd', 'dont_nombre_de_logements_pls', 'geo_point_2d', 'geo_shape']


,identifiant_livraison,annee_financement_agrement,arrondissement_num,bailleur_social,programme_type,nombre_total_de_logements_finances,dont_nombre_de_logements_pla_i,dont_nombre_de_logements_plus,dont_nombre_de_logements_plus_cd,dont_nombre_de_logements_pls,geo_point_2d,geo_shape
0,2000011,2001,12,RES.URB.,logement familial,62,0,62,0,0,"48.83399817959342, 2.398175442412243","{""coordinates"": [2.398175442412243, 48.83399817959342], ""type"": ""Point""}"
1,2000035,2001,19,RIVP,logement familial,18,10,8,0,0,"48.8783619998301, 2.3895183752327043","{""coordinates"": [2.3895183752327043, 48.8783619998301], ""type"": ""Point""}"
2,2000044,2001,20,RIVP,logement familial,6,0,6,0,0,"48.85500366971155, 2.403865260493333","{""coordinates"": [2.403865260493333, 48.85500366971155], ""type"": ""Point""}"


,taux_manquants
identifiant_livraison,0.0
annee_financement_agrement,0.0
arrondissement_num,0.0
bailleur_social,0.0
programme_type,0.0
nombre_total_de_logements_finances,0.0
dont_nombre_de_logements_pla_i,0.0
dont_nombre_de_logements_plus,0.0
dont_nombre_de_logements_plus_cd,0.0
dont_nombre_de_logements_pls,0.0


In [10]:
rent_caps["annee"] = pd.to_numeric(rent_caps["annee"], errors="coerce")
rent_caps["numero_du_quartier"] = pd.to_numeric(rent_caps["numero_du_quartier"], errors="coerce")
rent_caps["nombre_de_pieces_principales"] = pd.to_numeric(rent_caps["nombre_de_pieces_principales"], errors="coerce")
rent_caps["loyers_de_reference"] = pd.to_numeric(rent_caps["loyers_de_reference"], errors="coerce")
rent_caps["loyers_de_reference_majores"] = pd.to_numeric(rent_caps["loyers_de_reference_majores"], errors="coerce")
rent_caps["loyers_de_reference_minores"] = pd.to_numeric(rent_caps["loyers_de_reference_minores"], errors="coerce")
rent_caps["arrondissement_num"] = rent_caps["numero_insee_du_quartier"].astype(str).str[:5].str[-2:]
rent_caps["type_location"] = rent_caps["type_de_location"].astype(str)
rent_caps["quartier"] = rent_caps["nom_du_quartier"].astype(str)
rent_caps["ecart_majore_reference"] = rent_caps["loyers_de_reference_majores"] - rent_caps["loyers_de_reference"]
rent_caps["ecart_reference_minore"] = rent_caps["loyers_de_reference"] - rent_caps["loyers_de_reference_minores"]

rent_caps_clean = rent_caps[[
    "annee",
    "arrondissement_num",
    "numero_insee_du_quartier",
    "quartier",
    "nombre_de_pieces_principales",
    "epoque_de_construction",
    "type_location",
    "loyers_de_reference",
    "loyers_de_reference_majores",
    "loyers_de_reference_minores",
    "ecart_majore_reference",
    "ecart_reference_minore",
    "geo_point_2d",
    "geo_shape",
]].drop_duplicates()

profile_dataframe(rent_caps_clean, "Encadrement des loyers nettoyé")


===== Encadrement des loyers nettoyé =====
shape: (17920, 14)
colonnes:
['annee', 'arrondissement_num', 'numero_insee_du_quartier', 'quartier', 'nombre_de_pieces_principales', 'epoque_de_construction', 'type_location', 'loyers_de_reference', 'loyers_de_reference_majores', 'loyers_de_reference_minores', 'ecart_majore_reference', 'ecart_reference_minore', 'geo_point_2d', 'geo_shape']


,annee,arrondissement_num,numero_insee_du_quartier,quartier,nombre_de_pieces_principales,epoque_de_construction,type_location,loyers_de_reference,loyers_de_reference_majores,loyers_de_reference_minores,ecart_majore_reference,ecart_reference_minore,geo_point_2d,geo_shape
0,2023,20,7512077,Belleville,3,1946-1970,non meublé,19.8,23.8,13.9,4.0,5.9,"48.87153120058614, 2.3875492398498035","{""coordinates"": [[[2.3832266916750324, 48.86709755024514], [2.383136036636736, 48.86707551238079], [2.38309592865078..."
1,2023,06,7510621,Monnaie,2,Avant 1946,meublé,33.0,39.6,23.1,6.6,9.9,"48.85438440363996, 2.340035371130514","{""coordinates"": [[[2.3431685504395383, 48.851388550003655], [2.3426778705512987, 48.850357652304076], [2.34130585569..."
2,2023,03,7510312,Sainte-Avoie,1,1971-1990,meublé,34.8,41.8,24.4,7.0,10.4,"48.862557245043206, 2.354851518245669","{""coordinates"": [[[2.3582176804342043, 48.86122492296987], [2.356903330605401, 48.86006582105499], [2.35686231490314..."


,taux_manquants
annee,0.0
arrondissement_num,0.0
numero_insee_du_quartier,0.0
quartier,0.0
nombre_de_pieces_principales,0.0
epoque_de_construction,0.0
type_location,0.0
loyers_de_reference,0.0
loyers_de_reference_majores,0.0
loyers_de_reference_minores,0.0


## 2. Agrégation et fusion

On construit une table Gold par arrondissement et par année pour alimenter le dashboard:
- transactions immobilières,
- production de logement social,
- loyers encadrés.

La couche géographique reste séparée pour servir les polygones et les géométries.

In [11]:
transactions_agg = (
    transactions_clean.groupby(["arrondissement_num", "annee"], as_index=False)
    .agg(
        nb_transactions=("id_mutation", "nunique"),
        prix_median_m2=("prix_m2", "median"),
        prix_moyen_m2=("prix_m2", "mean"),
        surface_mediane=("surface_reelle_bati", "median"),
        prix_median=("valeur_fonciere", "median"),
    )
)

social_agg = (
    social_housing_clean.groupby(["arrondissement_num", "annee_financement_agrement"], as_index=False)
    .agg(
        nb_programmes_sociaux=("identifiant_livraison", "nunique"),
        logements_finances=("nombre_total_de_logements_finances", "sum"),
        logements_plai=("dont_nombre_de_logements_pla_i", "sum"),
        logements_plus=("dont_nombre_de_logements_plus", "sum"),
        logements_pls=("dont_nombre_de_logements_pls", "sum"),
    )
    .rename(columns={"annee_financement_agrement": "annee"})
)

rent_agg = (
    rent_caps_clean.groupby(["arrondissement_num", "annee"], as_index=False)
    .agg(
        loyer_reference_median=("loyers_de_reference", "median"),
        loyer_majorer_median=("loyers_de_reference_majores", "median"),
        loyer_minorer_median=("loyers_de_reference_minores", "median"),
        ecart_moyen_majore=("ecart_majore_reference", "mean"),
        ecart_moyen_minore=("ecart_reference_minore", "mean"),
    )
)

main_gold = (
    transactions_agg.merge(social_agg, on=["arrondissement_num", "annee"], how="outer")
    .merge(rent_agg, on=["arrondissement_num", "annee"], how="outer")
    .sort_values(["arrondissement_num", "annee"])
)

main_gold = main_gold.merge(
    arrondissements_clean[["arrondissement_num", "arrondissement_code_insee", "arrondissement_label", "surface", "perimetre"]],
    on="arrondissement_num",
    how="left",
)

profile_dataframe(main_gold, "Table Gold fusionnée")
main_gold.head(10)


===== Table Gold fusionnée =====
shape: (477, 21)
colonnes:
['arrondissement_num', 'annee', 'nb_transactions', 'prix_median_m2', 'prix_moyen_m2', 'surface_mediane', 'prix_median', 'nb_programmes_sociaux', 'logements_finances', 'logements_plai', 'logements_plus', 'logements_pls', 'loyer_reference_median', 'loyer_majorer_median', 'loyer_minorer_median', 'ecart_moyen_majore', 'ecart_moyen_minore', 'arrondissement_code_insee', 'arrondissement_label', 'surface', 'perimetre']


,arrondissement_num,annee,nb_transactions,prix_median_m2,prix_moyen_m2,surface_mediane,prix_median,nb_programmes_sociaux,logements_finances,logements_plai,logements_plus,logements_pls,loyer_reference_median,loyer_majorer_median,loyer_minorer_median,ecart_moyen_majore,ecart_moyen_minore,arrondissement_code_insee,arrondissement_label,surface,perimetre
0,01,2003,NaN,NaN,NaN,NaN,NaN,2.0,38.0,5.0,33.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
1,01,2005,NaN,NaN,NaN,NaN,NaN,3.0,19.0,8.0,11.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
2,01,2007,NaN,NaN,NaN,NaN,NaN,5.0,90.0,24.0,44.0,22.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862


,taux_manquants
nb_transactions,0.958071
prix_moyen_m2,0.958071
prix_median_m2,0.958071
prix_median,0.958071
surface_mediane,0.958071
ecart_moyen_minore,0.706499
ecart_moyen_majore,0.706499
loyer_reference_median,0.706499
loyer_majorer_median,0.706499
loyer_minorer_median,0.706499


,arrondissement_num,annee,nb_transactions,prix_median_m2,prix_moyen_m2,surface_mediane,prix_median,nb_programmes_sociaux,logements_finances,logements_plai,logements_plus,logements_pls,loyer_reference_median,loyer_majorer_median,loyer_minorer_median,ecart_moyen_majore,ecart_moyen_minore,arrondissement_code_insee,arrondissement_label,surface,perimetre
0,01,2003,NaN,NaN,NaN,NaN,NaN,2.0,38.0,5.0,33.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
1,01,2005,NaN,NaN,NaN,NaN,NaN,3.0,19.0,8.0,11.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
2,01,2007,NaN,NaN,NaN,NaN,NaN,5.0,90.0,24.0,44.0,22.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
3,01,2008,NaN,NaN,NaN,NaN,NaN,23.0,207.0,4.0,203.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
4,01,2009,NaN,NaN,NaN,NaN,NaN,1.0,7.0,7.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
5,01,2010,NaN,NaN,NaN,NaN,NaN,5.0,52.0,14.0,35.0,3.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
6,01,2011,NaN,NaN,NaN,NaN,NaN,6.0,147.0,76.0,59.0,12.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
7,01,2012,NaN,NaN,NaN,NaN,NaN,3.0,32.0,7.0,25.0,0.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
8,01,2013,NaN,NaN,NaN,NaN,NaN,4.0,117.0,28.0,59.0,30.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862
9,01,2014,NaN,NaN,NaN,NaN,NaN,3.0,29.0,11.0,13.0,5.0,NaN,NaN,NaN,NaN,NaN,75101,Louvre,1.824613e+06,6054.936862


## 3. Indicateurs prêts pour le dashboard

Indicateurs déjà calculables à partir des CSV:
- prix médian au m² par arrondissement et par année,
- nombre de transactions et surface médiane,
- volume de logements sociaux financés et ventilation PLAI / PLUS / PLS,
- loyer de référence médian et écart avec le plafond majoré,
- comparaison entre deux arrondissements sur une même année.

In [12]:
kpis = {
    "prix_median_m2_global": transactions_clean["prix_m2"].median(),
    "transactions_total": transactions_clean["id_mutation"].nunique(),
    "logements_sociaux_total": social_housing_clean["nombre_total_de_logements_finances"].sum(),
    "loyer_reference_median_global": rent_caps_clean["loyers_de_reference"].median(),
}

pd.Series(kpis).to_frame("valeur")

,valeur
prix_median_m2_global,10967.152012
transactions_total,42875.000000
logements_sociaux_total,123709.000000
loyer_reference_median_global,26.100000


## 4. Prochaines étapes

1. Brancher ces tables Silver/Gold sur une API.
2. Ajouter les données externes manquantes si tu veux un indicateur revenus ou sécurité.
3. Générer les visuels: carte choroplèthe, points, timeline et comparaison de deux arrondissements.
4. Exporter les tables propres en parquet ou en base SQL pour le dashboard.

## 5. Enrichissement territorial optionnel

Le fichier data.gouv est très volumineux. On le traite en streaming par chunks pour ne garder que Paris et préparer un indicateur de cadre de vie ou de sécurité.

In [13]:
territorial_columns = ["CODGEO_2025", "annee", "indicateur", "unite_de_compte", "nombre", "taux_pour_mille", "est_diffuse"]
territorial_chunks = []

for chunk in pd.read_csv(
    RAW_FILES["territorial_stats"],
    sep=";",
    encoding="utf-8",
    usecols=territorial_columns,
    chunksize=500_000,
    low_memory=False,
):
    chunk = chunk.loc[chunk["CODGEO_2025"].astype(str).str.startswith("75")].copy()
    chunk["annee"] = pd.to_numeric(chunk["annee"], errors="coerce")
    chunk["arrondissement_num"] = chunk["CODGEO_2025"].astype(str).str[-2:]
    chunk["nombre"] = pd.to_numeric(
        chunk["nombre"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    chunk["taux_pour_mille"] = pd.to_numeric(
        chunk["taux_pour_mille"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    territorial_chunks.append(chunk)

territorial_paris = pd.concat(territorial_chunks, ignore_index=True)
territorial_agg = (
    territorial_paris.groupby(["arrondissement_num", "annee", "indicateur"], as_index=False)
    .agg(
        nombre_total=("nombre", "sum"),
        taux_moyen=("taux_pour_mille", "mean"),
    )
    .sort_values(["arrondissement_num", "annee", "indicateur"])
)

territorial_agg.head(10)

,arrondissement_num,annee,indicateur,nombre_total,taux_moyen
0,01,2016,Cambriolages de logement,134.0,9.712028
1,01,2016,Destructions et dégradations volontaires,415.0,25.535319
2,01,2016,Escroqueries et fraudes aux moyens de paiement,530.0,32.611371
3,01,2016,Trafic de stupéfiants,42.0,2.584297
4,01,2016,Usage de stupéfiants,607.0,37.349249
5,01,2016,Usage de stupéfiants (AFD),0.0,0.000000
6,01,2016,Violences physiques hors cadre familial,305.0,18.766921
7,01,2016,Violences physiques intrafamiliales,35.0,2.153581
8,01,2016,Violences sexuelles,82.0,5.045533
9,01,2016,Vols avec armes,31.0,1.907458


### Résultat attendu

À la fin de l'exécution, tu obtiens:
- des tables Silver propres par source,
- une table Gold fusionnée par arrondissement et par année,
- des indicateurs de prix, loyers, social et éventuellement sécurité,
- une base propre à brancher sur une API ou un front cartographique.

## 6. Quatre indicateurs obligatoires consolidés

Cette dernière table rassemble les 4 indicateurs retenus dans un seul dataset exploitable pour le dashboard.

In [14]:
security_focus = [
    "Cambriolages de logement",
    "Destructions et dégradations volontaires",
    "Escroqueries et fraudes aux moyens de paiement",
    "Trafic de stupéfiants",
    "Usage de stupéfiants",
    "Violences physiques hors cadre familial",
    "Violences physiques intrafamiliales",
    "Violences sexuelles",
    "Vols avec armes",
    "Vols violents sans arme",
    "Vols sans violence contre des personnes",
]

security_indicators = (
    territorial_agg.loc[territorial_agg["indicateur"].isin(security_focus)]
    .groupby(["arrondissement_num", "annee"], as_index=False)
    .agg(
        incidents_securite=("nombre_total", "sum"),
        taux_securite_moyen=("taux_moyen", "mean"),
    )
)

mandatory_indicators = (
    main_gold.merge(security_indicators, on=["arrondissement_num", "annee"], how="left")
    .assign(
        ecart_loyer_reference_majoration=lambda df: df["loyer_majorer_median"] - df["loyer_reference_median"],
        part_logements_plai=lambda df: np.where(
            df["logements_finances"] > 0,
            df["logements_plai"] / df["logements_finances"],
            np.nan,
        ),
    )
    .sort_values(["arrondissement_num", "annee"])
)

mandatory_indicators = mandatory_indicators[[
    "arrondissement_num",
    "arrondissement_label",
    "annee",
    "prix_median_m2",
    "nb_transactions",
    "loyer_reference_median",
    "loyer_majorer_median",
    "ecart_loyer_reference_majoration",
    "logements_finances",
    "part_logements_plai",
    "incidents_securite",
    "taux_securite_moyen",
]]

mandatory_indicators.head(20)

,arrondissement_num,arrondissement_label,annee,prix_median_m2,nb_transactions,loyer_reference_median,loyer_majorer_median,ecart_loyer_reference_majoration,logements_finances,part_logements_plai,incidents_securite,taux_securite_moyen
0,01,Louvre,2003,NaN,NaN,NaN,NaN,NaN,38.0,0.131579,NaN,NaN
1,01,Louvre,2005,NaN,NaN,NaN,NaN,NaN,19.0,0.421053,NaN,NaN
2,01,Louvre,2007,NaN,NaN,NaN,NaN,NaN,90.0,0.266667,NaN,NaN
3,01,Louvre,2008,NaN,NaN,NaN,NaN,NaN,207.0,0.019324,NaN,NaN
4,01,Louvre,2009,NaN,NaN,NaN,NaN,NaN,7.0,1.000000,NaN,NaN
5,01,Louvre,2010,NaN,NaN,NaN,NaN,NaN,52.0,0.269231,NaN,NaN
6,01,Louvre,2011,NaN,NaN,NaN,NaN,NaN,147.0,0.517007,NaN,NaN
7,01,Louvre,2012,NaN,NaN,NaN,NaN,NaN,32.0,0.218750,NaN,NaN
8,01,Louvre,2013,NaN,NaN,NaN,NaN,NaN,117.0,0.239316,NaN,NaN
9,01,Louvre,2014,NaN,NaN,NaN,NaN,NaN,29.0,0.379310,NaN,NaN


In [21]:
mandatory_indicators.count()

arrondissement_num                  477
arrondissement_label                477
annee                               477
prix_median_m2                       20
nb_transactions                      20
loyer_reference_median              140
loyer_majorer_median                140
ecart_loyer_reference_majoration    140
logements_finances                  447
part_logements_plai                 447
incidents_securite                  199
taux_securite_moyen                 199
dtype: int64

In [15]:
indicator_kpis = pd.DataFrame(
    {
        "indicateur": [
            "Prix médian au m²",
            "Loyer de référence médian",
            "Logements sociaux financés",
            "Sécurité / cadre de vie",
        ],
        "valeur_globale": [
            transactions_clean["prix_m2"].median(),
            rent_caps_clean["loyers_de_reference"].median(),
            social_housing_clean["nombre_total_de_logements_finances"].sum(),
            security_indicators["taux_securite_moyen"].mean(),
        ],
    }
)

display(indicator_kpis)
display(mandatory_indicators.head(25))

,indicateur,valeur_globale
0,Prix médian au m²,10967.152012
1,Loyer de référence médian,26.100000
2,Logements sociaux financés,123709.000000
3,Sécurité / cadre de vie,14.647788


,arrondissement_num,arrondissement_label,annee,prix_median_m2,nb_transactions,loyer_reference_median,loyer_majorer_median,ecart_loyer_reference_majoration,logements_finances,part_logements_plai,incidents_securite,taux_securite_moyen
0,01,Louvre,2003,NaN,NaN,NaN,NaN,NaN,38.0,0.131579,NaN,NaN
1,01,Louvre,2005,NaN,NaN,NaN,NaN,NaN,19.0,0.421053,NaN,NaN
2,01,Louvre,2007,NaN,NaN,NaN,NaN,NaN,90.0,0.266667,NaN,NaN
3,01,Louvre,2008,NaN,NaN,NaN,NaN,NaN,207.0,0.019324,NaN,NaN
4,01,Louvre,2009,NaN,NaN,NaN,NaN,NaN,7.0,1.000000,NaN,NaN
5,01,Louvre,2010,NaN,NaN,NaN,NaN,NaN,52.0,0.269231,NaN,NaN
6,01,Louvre,2011,NaN,NaN,NaN,NaN,NaN,147.0,0.517007,NaN,NaN
7,01,Louvre,2012,NaN,NaN,NaN,NaN,NaN,32.0,0.218750,NaN,NaN
8,01,Louvre,2013,NaN,NaN,NaN,NaN,NaN,117.0,0.239316,NaN,NaN
9,01,Louvre,2014,NaN,NaN,NaN,NaN,NaN,29.0,0.379310,NaN,NaN


### Lecture rapide des 4 indicateurs

- Prix médian au m²: mesure du marché de l’achat.
- Loyer de référence médian et écart de majoration: mesure de tension locative.
- Logements sociaux financés: mesure de l’effort public.
- Sécurité / cadre de vie: mesure de contexte territorial.